# Лабораторная работа №3
## Решение задач многомерной условной оптимизации методом множителей Лагранжа

**Выполнила:** Федорова Мария Витальевна  
**Вариант:** 12

**Цель работы:** освоить метод множителей Лагранжа для поиска условных экстремумов функций нескольких переменных, реализовать метод программно и проверить результаты аналитически

In [30]:
import sympy as sp
sp.init_printing()

In [31]:
x,y = sp.symbols("x y", real=True)
variables = [x, y]

objective = x**2 + y**2
constraints = [x * y - 1]

print("Целевая функция:")
sp.pprint(objective)
print("\nОграничения:")
for constraint in constraints:
    sp.pprint(sp.Eq(constraint, 0))


Целевая функция:
 2    2
x  + y 

Ограничения:
x⋅y - 1 = 0


In [32]:

multipliers = sp.symbols(
    f"lambda_1:{len(constraints) + 1}",
    real=True,
)
lagrangian = objective + sum(
    multiplier * constraint
    for multiplier, constraint in zip(multipliers, constraints)
)
stationarity_equations = [
    sp.diff(lagrangian, variable)
    for variable in variables
]
equations = stationarity_equations + constraints
unknowns = variables + list(multipliers)

print("Функция Лагранжа:")
sp.pprint(lagrangian)
print("\nСистема уравнений:")
for equation in equations:
    sp.pprint(sp.Eq(equation, 0))


Функция Лагранжа:
                2    2
λ₁⋅(x⋅y - 1) + x  + y 

Система уравнений:
λ₁⋅y + 2⋅x = 0
λ₁⋅x + 2⋅y = 0
x⋅y - 1 = 0



## Ручное решение системы Лагранжа

Функция Лагранжа:

$$
\mathcal{L}(x,y,\lambda)=x^2+y^2+\lambda(xy-1)
$$

Приравниваем частные производные к нулю и добавляем исходное ограничение:

$$
\begin{cases}
2x+\lambda y=0\\
2y+\lambda x=0\\
xy=1
\end{cases}
$$

Из ограничения $xy=1$ следует, что $x\neq0$ и $y\neq0$

Умножим первое уравнение на $x$, а второе на $y$:

$$
\begin{cases}
2x^2+\lambda xy=0\\
2y^2+\lambda xy=0
\end{cases}
$$

Вычитая уравнения, получаем:

$$
2x^2-2y^2=0
$$

$$
x^2=y^2
$$

Отсюда $x=y$ или $x=-y$

Случай $x=-y$ невозможен, поскольку тогда $xy=-x^2\leq0$, что противоречит ограничению $xy=1$

Следовательно, $x=y$, и из ограничения получаем:

$$
x^2=1
$$

Таким образом, найдены две точки:

$$
(x_1,y_1)=(-1,-1)
$$

$$
(x_2,y_2)=(1,1)
$$

Найдём множитель Лагранжа из первого уравнения:

$$
\lambda=-\frac{2x}{y}=-2
$$

Для обеих точек $\lambda=-2$

Вычислим значения целевой функции:

$$
f(-1,-1)=(-1)^2+(-1)^2=2
$$

$$
f(1,1)=1^2+1^2=2
$$

Других действительных решений системы нет, поскольку оба возможных случая $x^2=y^2$ рассмотрены


In [33]:

jacobian = sp.Matrix(constraints).jacobian(variables)
print("Матрица Якоби ограничений: ")
sp.pprint(jacobian)
singular_solutions = sp.solve(
    constraints + list(jacobian),
    variables,
    dict=True,
)
print("\nОсобые допустимые точки: ")
print(singular_solutions if singular_solutions else "Не найдены ")


Матрица Якоби ограничений: 
[y  x]

Особые допустимые точки: 
Не найдены 


### Анализ применимости метода

целевая функция $f(x,y)=x^2+y^2$ и ограничение $h(x,y)=xy-1$ определены и непрерывно дифференцируемы на всей плоскости $\mathbb{R}^2$
допустимое множество непусто, поскольку точка $(1,1)$ удовлетворяет ограничению $xy=1$

градиент ограничения:

$$
\nabla h(x,y)=(y,x)
$$

он обращается в ноль только при $x=y=0$, однако эта точка не удовлетворяет ограничению $xy=1$
=> градиент ограничения не равен нулю во всех допустимых точках, а матрица Якоби имеет ранг 1
таким образом, ограничение регулярно и метод множителей Лагранжа применим для поиска точек-кандидатов на условный экстремум


In [34]:

solutions = sp.solve(equations, unknowns, dict=True)
print(f"Количество найденных решений: {len(solutions)}")
for index, solution in enumerate(solutions, start=1):
    print(f"\nКандидат {index}:")
    for unknown in unknowns:
        print(f"  {unknown} = {sp.simplify(solution[unknown])}")


Количество найденных решений: 2

Кандидат 1:
  x = -1
  y = -1
  lambda_1 = -2

Кандидат 2:
  x = 1
  y = 1
  lambda_1 = -2


In [35]:

tolerance = 1e-8
candidates = []

for solution in solutions:
    point = {
        variable: sp.simplify(solution[variable])
        for variable in variables
    }

    residuals = [
        sp.simplify(constraint.subs(point))
        for constraint in constraints
    ]

    stationary_residuals = [
        sp.simplify(equation.subs(solution))
        for equation in equations
    ]

    is_real = all(
        value.is_real is True
        for value in point.values()
    )

    is_feasible = is_real and all(
        abs(float(sp.N(residual))) <= tolerance
        for residual in residuals
    )

    is_stationary = all(
        abs(float(sp.N(residual))) <= tolerance
        for residual in stationary_residuals
    )

    objective_value = sp.simplify(objective.subs(point))

    candidates.append({
        "point": point,
        "multipliers": {
            multiplier: sp.simplify(solution[multiplier])
            for multiplier in multipliers
        },
        "value": objective_value,
        "residuals": residuals,
        "feasible": is_feasible,
        "stationary": is_stationary,
    })

for index, candidate in enumerate(candidates, start=1):
    print(f"\nКандидат {index}")
    print("Точка: ", candidate["point"])
    print("Множители Лагранжа: ", candidate["multipliers"])
    print("Значение функции: " , candidate["value"])
    print("Невязки ограничений: ", candidate["residuals"])
    print("Допустимость: ", "Да" if candidate["feasible"] else "Нет")
    print("Стационарность: ", "Да" if candidate["stationary"] else "Нет")



Кандидат 1
Точка:  {x: -1, y: -1}
Множители Лагранжа:  {lambda_1: -2}
Значение функции:  2
Невязки ограничений:  [0]
Допустимость:  Да
Стационарность:  Да

Кандидат 2
Точка:  {x: 1, y: 1}
Множители Лагранжа:  {lambda_1: -2}
Значение функции:  2
Невязки ограничений:  [0]
Допустимость:  Да
Стационарность:  Да



## Обоснование глобальных экстремумов

метод множителей Лагранжа позволяет найти точки-кандидаты, но сам по себе не доказывает глобальность экстремумов

для нашего ограничения $xy=1$ используем неравенство:

$$
x^2+y^2 \geq 2|xy|=2
$$

=> значение целевой функции не может быть меньше 2

в точках $(1,1)$ и $(-1,-1)$ значение функции равно 2. значит,обе точки являются глобальными минимумами

глобального максимума не существует. действительно, при $x=t$, $y=\frac{1}{t}$ ограничение выполняется, а

$$
f\left(t,\frac{1}{t}\right)=t^2+\frac{1}{t^2}\longrightarrow+\infty
$$

при $t\longrightarrow+\infty$

таким образом функция не ограничена сверху на допустимом множестве


In [36]:

valid_candidates = [
    candidate
    for candidate in candidates
    if candidate["feasible"] and candidate["stationary"]
]
if not valid_candidates:
    raise RuntimeError("Допустимые кандидаты не найдены ")
minimum_value = min(
    candidate["value"]
    for candidate in valid_candidates
)
minimum_points = [
    candidate["point"]
    for candidate in valid_candidates
    if sp.simplify(candidate["value"] - minimum_value) == 0
]

print("Глобальный минимум функции: ", minimum_value)
print("Точки глобального минимума: ")
for point in minimum_points:
    print(tuple(point[variable] for variable in variables))
print("Аналитический вывод: глобального максимума нет")
print("Обоснование: при x = t и y = 1/t функция не ограничена сверху")


Глобальный минимум функции:  2
Точки глобального минимума: 
(-1, -1)
(1, 1)
Аналитический вывод: глобального максимума нет
Обоснование: при x = t и y = 1/t функция не ограничена сверху



## Сравнение ручного и программного решений

| Характеристика | Ручное решение | Программное решение |
|---|---|---|
| Точки-кандидаты | $(-1,-1)$, $(1,1)$ | $(-1,-1)$, $(1,1)$ |
| Множитель Лагранжа | $\lambda=-2$ для обеих точек | $\lambda=-2$ для обеих точек |
| Значения функции | $f(-1,-1)=2$, $f(1,1)=2$ | $2$, $2$ |
| Невязки ограничений | $0$ для обеих точек | $0$ для обеих точек |
| Глобальный минимум | $f_{\min}=2$ | $f_{\min}=2$ |
| Точки глобального минимума | $(-1,-1)$, $(1,1)$ | $(-1,-1)$, $(1,1)$ |
| Глобальный максимум | Не существует | Не существует |

ручное и программное решения полностью совпадают

обе найденные точки удовлетворяют исходному ограничению и являются глобальными минимумами



## Вывод

в ходе лабораторной работы был изучен и программно реализован метод множителей Лагранжа для решения задачи многомерной условной оптимизации

была проверена применимость метода: целевая функция и ограничение непрерывно дифференцируемы, допустимое множество непусто,а ограничение регулярно во всех допустимых точках

программа автоматически построила функцию Лагранжа, вычислила частные производные, составила систему уравнений и нашла две точки-кандидата: $(-1,-1)$ и $(1,1)$

для обеих точек значение целевой функции равно $2$, множитель Лагранжа равен $-2$, а невязки ограничений равны нулю

дополнительно было доказано, что обе точки являются глобальными минимумами, поскольку на допустимом множестве выполняется неравенство $x^2+y^2\geq2$

глобального максимума не существует,тк целевая функция не ограничена сверху при условии $xy=1$

результаты ручного и программного решений полностью совпали

основное ограничение метода заключается в том, что решение системы Лагранжа даёт только точки-кандидаты, поэтому глобальность экстремумов необходимо обосновывать отдельно
